In [25]:
from typing import TypedDict, Annotated, List
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import MemorySaver

# 1. 定义状态
class State(TypedDict):
    messages: Annotated[List, add_messages]

# 2. 初始化模型
model = ChatOpenAI(
    model="qwen3-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key="sk-8b293e101d5e41f1b3116d9848b7ac59",
    streaming=True
)
# 3. 定义节点
def chatbot(state: State) -> dict:
    response = model.invoke(state["messages"])
    print("response:", response)
    return {"messages": [response]}

# 4. 构建图
graph = StateGraph(State)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

# 5. 编译（带记忆）
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

# 6. 使用
config = {"configurable": {"thread_id": "conversation-1"}}

# 第一轮
result = app.invoke({"messages": [HumanMessage(content="你好！")]}, config)
print(result["messages"][-1].content)

# 第二轮（记住上下文）
generator = app.stream({"messages": [HumanMessage(content="我叫小明，你呢？")]}, config)
for item in generator:
    print(item)
# print(result["messages"][-1].content)

response: content='你好！很高兴见到你，有什么我可以帮忙的吗？😊' additional_kwargs={} response_metadata={'finish_reason': 'stop', 'model_name': 'qwen3-max', 'model_provider': 'openai'} id='lc_run--019d3e6a-9f6d-7ed3-9311-34d2885a3fdd' tool_calls=[] invalid_tool_calls=[]
你好！很高兴见到你，有什么我可以帮忙的吗？😊
response: content='你好，小明！我叫通义千问，英文名是Qwen。我是阿里巴巴集团研发的超大规模语言模型，可以回答问题、创作文字、编程等等。你可以叫我千问或者Qwen～  \n很高兴认识你！有什么我可以帮你的吗？😊' additional_kwargs={} response_metadata={'finish_reason': 'stop', 'model_name': 'qwen3-max', 'model_provider': 'openai'} id='lc_run--019d3e6a-a2e2-77d0-b814-003bbe033328' tool_calls=[] invalid_tool_calls=[]
{'chatbot': {'messages': [AIMessage(content='你好，小明！我叫通义千问，英文名是Qwen。我是阿里巴巴集团研发的超大规模语言模型，可以回答问题、创作文字、编程等等。你可以叫我千问或者Qwen～  \n很高兴认识你！有什么我可以帮你的吗？😊', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'qwen3-max', 'model_provider': 'openai'}, id='lc_run--019d3e6a-a2e2-77d0-b814-003bbe033328', tool_calls=[], invalid_tool_calls=[])]}}


In [23]:
from typing import TypedDict, Annotated, List
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import MemorySaver

# 1. 定义状态
class State(TypedDict):
    messages: Annotated[List, add_messages]

# 2. 初始化模型（启用流式）
model = ChatOpenAI(
    model="qwen3-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key="sk-8b293e101d5e41f1b3116d9848b7ac59",
    streaming=True  # 这个参数主要影响底层 behavior，但仍建议配合 .stream() 使用
)

# 3. 定义节点（保持不变）
def chatbot(state: State) -> dict:
    response = model.invoke(state["messages"])
    return {"messages": [response]}

# 4. 构建图
graph = StateGraph(State)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

# 5. 编译（带记忆）
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

# 6. 使用：改为流式输出
config = {"configurable": {"thread_id": "conversation-1"}}

# 第一轮对话：发送消息并流式接收输出
for event in app.stream(
    {"messages": [HumanMessage(content="你好！")]},
    config=config,
    stream_mode="values"  # 可选 'values', 'updates', 'debug' 等
):
    # event 是一个完整的 state 快照（因为 stream_mode="values"）
    message = event["messages"][-1]
    if isinstance(message, AIMessage) and hasattr(message, "content") and message.content:
        # 打印新增的内容（增量模式下更高效，这里简单处理）
        print(message.content, end="\n", flush=True)
print()  # 换行


你好！很高兴见到你，有什么我可以帮忙的吗？😊

